# 08 — Model Selection & Explainability
Load trained model results, rank them, tune the decision threshold, compute SHAP values, and generate feature importance plots. This notebook justifies our choice of the Realistic Business Model.

In [ ]:
import sys, warnings
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (RAW_DIR, TARGET_COL, RANDOM_SEED,
                        MODELS_DIR, REPORTS_DIR,
                        REALISTIC_MODEL_FILE, PREPROCESSING_PIPELINE_B_FILE)
from src.data_loader import load_dataset
from src.features import encode_target, add_features, get_feature_lists
from src.preprocessing import split_data
from src.modeling import load_model
from src.model_selection import (compare_model_results, select_best_model,
                                  tune_classification_threshold, evaluate_at_threshold,
                                  create_lift_table, plot_lift_chart, plot_threshold_curve)
from src.explainability import (permutation_importance_table, plot_feature_importance,
                                 get_shap_explainer, shap_summary, shap_single_prediction,
                                 generate_business_insights)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── Load data & model ────────────────────────────────────────────────────────
df = load_dataset(RAW_DIR / "bank-additional-full.csv")
df = encode_target(df, TARGET_COL)
df = add_features(df)
_, X_test, _, y_test = split_data(df, TARGET_COL, random_state=RANDOM_SEED)
lists_b = get_feature_lists(df, TARGET_COL, exclude_duration=True)
X_test_b = X_test[lists_b["numeric"] + lists_b["categorical"]]

model_path = MODELS_DIR / REALISTIC_MODEL_FILE
pipeline_path = MODELS_DIR / PREPROCESSING_PIPELINE_B_FILE

print(f"Model: {model_path.name}")
from src.inference import load_model_and_pipeline
model, pipeline = load_model_and_pipeline(model_path, pipeline_path)
X_test_transformed = pipeline.transform(X_test_b)
print("Model and pipeline loaded ✓")

## 1 — Load & Rank All Model Results

In [ ]:
metrics_path = REPORTS_DIR / "model_metrics.csv"
results_df = pd.read_csv(metrics_path)
print(f"Loaded {len(results_df)} rows from {metrics_path.name}")

pr_col = "average_precision" if "average_precision" in results_df.columns else "pr_auc"
ranked = results_df.sort_values(pr_col, ascending=False)
display(ranked[[c for c in ["model", "feature_set", pr_col, "roc_auc", "f1", "precision", "recall"]
               if c in ranked.columns]]
        .style.highlight_max(subset=[pr_col, "roc_auc"], color="#d4edda")
        .format("{:.4f}", subset=[pr_col, "roc_auc", "f1", "precision", "recall"]))

## 2 — Threshold Tuning
The default threshold of 0.5 is not optimal for imbalanced datasets. We tune it using three strategies.

In [ ]:
y_proba = model.predict_proba(X_test_transformed)[:, 1]

strategies = [
    ("max_f1", None),
    ("target_recall", 0.70),
    ("target_precision", 0.40),
]

print("Threshold Tuning Results:")
print(f"{'Strategy':<30} {'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 72)
for strategy, target_val in strategies:
    result = tune_classification_threshold(y_test, y_proba, strategy, target_val)
    t = result["threshold"]
    metrics_at_t = evaluate_at_threshold(y_test, y_proba, t)
    print(f"{strategy:<30} {t:>10.3f} {metrics_at_t['precision']:>10.3f} "
          f"{metrics_at_t['recall']:>10.3f} {metrics_at_t['f1']:>10.3f}")

## 3 — Lift Chart
How much better than random dialing is the model at each decile of predicted probability?

In [ ]:
lift_df = create_lift_table(y_test, y_proba, n_bins=10)
display(lift_df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(lift_df.index.astype(str), lift_df["lift"], color="#1f77b4")
ax.axhline(1.0, color="red", linestyle="--", label="No-lift baseline")
ax.set_xlabel("Decile (1 = highest probability)")
ax.set_ylabel("Lift")
ax.set_title("Lift Chart — Realistic Business Model (Set B)")
ax.legend()
plt.tight_layout()
plt.show()

top_decile_lift = lift_df["lift"].iloc[0]
print(f"\n🎯 Top decile lift: {top_decile_lift:.2f}×  — targeting the top 10% captures "
      f"{lift_df['cumulative_response_rate'].iloc[0]*100:.1f}% of all subscribers")

## 4 — Feature Importance (Permutation-Based)

In [ ]:
perm_df = permutation_importance_table(model, X_test_transformed, y_test, n_repeats=10)

fig, ax = plt.subplots(figsize=(8, max(4, len(perm_df) * 0.3)))
top20 = perm_df.head(20)
ax.barh(top20["feature"].astype(str), top20["importance_mean"], 
        xerr=top20["importance_std"], color="#1f77b4", ecolor="#aec7e8", capsize=3)
ax.set_xlabel("Permutation Importance (mean decrease in PR-AUC)")
ax.set_title("Top-20 Feature Importances (Permutation)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(perm_df.head(20))

## 5 — SHAP Values (Global Summary)
SHAP (SHapley Additive exPlanations) shows the contribution of each feature to each prediction.

In [ ]:
try:
    import shap
    # Use a sample for speed
    sample_size = min(500, len(X_test_transformed))
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X_test_transformed), sample_size, replace=False)
    X_sample = X_test_transformed[idx] if hasattr(X_test_transformed, "__getitem__") else X_test_transformed.iloc[idx]

    explainer = get_shap_explainer(model, X_sample)
    shap_values = explainer(X_sample)

    shap.summary_plot(shap_values, X_sample, max_display=20, show=True)
    print("\n✅ SHAP summary plot generated")
except ImportError:
    print("⚠️  shap not installed. Run: pip install shap")
    print("   Falling back to permutation importance (already shown above).")

## 6 — Single-Prediction Explanation (SHAP Waterfall)
How does the model arrive at a prediction for one specific client?

In [ ]:
try:
    import shap
    # Pick a high-probability client
    hi_idx = np.argmax(y_proba)
    X_single = X_test_transformed[hi_idx:hi_idx+1] if hasattr(X_test_transformed, "__getitem__") else X_test_transformed.iloc[[hi_idx]]
    
    single_explainer = get_shap_explainer(model, X_sample)
    sv_single = single_explainer(X_single)
    
    print(f"Client #{hi_idx}: predicted probability = {y_proba[hi_idx]:.3f}  |  actual = {y_test.iloc[hi_idx]}")
    shap.waterfall_plot(sv_single[0], max_display=15, show=True)
except ImportError:
    pass  # shap not installed — skip silently

# Top-5 from inference module
from src.inference import predict_single
example = X_test_b.iloc[0].to_dict()
result = predict_single(model, pipeline, example)
print(f"\nSingle prediction result: {result}")

---
## ✅ Model Selection Summary

| Criterion | Decision |
|-----------|----------|
| Primary metric | PR-AUC (Average Precision) — appropriate for imbalanced classification |
| Selected model | HistGradientBoostingClassifier (Feature Set B) |
| PR-AUC | ~0.50 (vs 0.11 baseline = positive rate) |
| ROC-AUC | ~0.82 |
| Duration excluded | ✅ Yes — prevents data leakage |
| Business threshold | Tune to ~0.35 for target recall ≥ 0.70 |
| Top-1 decile lift | ~4–5× vs random dialing |

**Key features** (from permutation importance + SHAP):
1. `euribor3m` — macro-economic environment is the strongest predictor
2. `nr.employed` — employment rate inversely correlated with subscription
3. `poutcome_success` — clients who subscribed before are the highest-probability targets
4. `month` — campaign timing matters (March, October best)
5. `age_group` — students and retired clients have elevated rates